In [ ]:
# Dependent Modules for Notebook
import numpy as np
from numpy import polyval
import matplotlib.pyplot as plt

# Low-Fidelity Estimation of Battery State of Charge

## Given

Discharge Curves - Coefficients of N-Polynomial Regression Fit
* Open-Circuit Voltage: \[-7.306e-17, 3.781e-13, -7.146e-10, 6.261e-07, -0.001, 4.202 \]
* Internal Resistance [INR26650-50A](https://www.betterbatterydesign.com/blog/2020/11/14/dc-ir-vs-soc): \[ 1374.205, -8490.740, 22555.209, -33702.304, 31118.442, -18370.550, 6938.585, -1632.785, 225.962, -16.519, 0.521 \]

> **WARNING** The open-circuit discharge curves, which serve as parameters to the model, are not representative of the current swift 021 battery; however, they are sufficient place-holders in demonstrating the battery model and providing a range of values for mathematical validation. These discharge curves come from a previous 021 hybrid battery given by the battery team(~08/2018) where the battery tests were ran on the CHROMA at different constant loads until the minimum cell voltage(~2.6V).





Open-Circuit Voltage,$OCV$, can be approximated by the curve:

$$ OCV = f(Q_{consummed}) $$
such that
$$ OCV(Q_{consummed}) = a_0 + a_1 \cdot Q_{consummed} + a_2 \cdot Q_{consummed}^2 + ... + a_n \cdot Q_{consummed}^n $$

In [ ]:
# load open circuit voltage discharge curve coefficients, curve is in mA consumed (mAh) vs. voltage
p_dc = [-7.3058069381599e-17, 3.78145139174554e-13, -7.14597902097917e-10, 6.26071232688909e-07, -0.000643100918689175, 4.20180995475114]

Q_consummed_mAh = np.linspace(0, 3400, 500)
voltage_V = polyval(p_dc, Q_consummed_mAh)
plt.plot( Q_consummed_mAh, voltage_V)
plt.grid()
plt.title('open circuit voltage discharge curve')
plt.legend()
plt.xlabel('Charge Consummed (mAh)')
plt.ylabel('Voltage (V)')
plt.show()


Internal resistance can be approximated by the curve:

$$ R_{internal} = f(SOC) $$
such that
$$ R_{internal}(SoC) = a_0 + a_1 \cdot SoC + a_2 \cdot SoC^2 + ... + a_n \cdot SoC^n $$

In [ ]:
# load internal resistance discharge curve coefficients, curve is in decimal state of charge vs. ohms
p_ir = [1374.20478941936, -8490.74044658212, 22555.2090615421, -33702.3045169079, 31118.4418493407, -18370.5497580968, 6938.58530566924, -1632.78483786197, 225.962184259985, -16.5195364175862, 0.521273828846724];

soc = np.linspace(0, 1, 100)
ir_Ohm = polyval(p_ir, soc)
plt.plot( soc, ir_Ohm)
plt.grid()
plt.title('internal resistance discharge curve')
plt.xlabel('SoC (mA)')
plt.ylabel('Internal Resistance (Ohm)')
plt.xlim(1.0, 0)
plt.show()


## Find 
Battery
* State of Charge
* Open Circuit Voltage
* Internal Resistance
* Voltage Seen by Load (Voltage with SAG)

For each case:
* 0.46A constant load for 150 steps of 1 minute/step
* 0.46A constant load for 300 steps of 1 minute/step
* 15A constant load for 4 steps of 1 minute/step
* 15A constant load for 7 steps of 1 minute/step
* 35A constant load for 100 steps of 1 second/step
* 35A constant load for 180 steps of 1 second/step

## Assumption
* Battery Capacity: 3.4 Ah
* Harness Resistance: 0.040 Ohms
* Number of Cells: 12

In [ ]:
# From Assumption
Q_rated = 3.4       # Rated Capacity of the Battery, mAh
R_harness = 0.040   # Resistance of the Harness, Ohms
num_cells = 12.0    # number of cells

## Analysis

We calculate battery state of charge through coloumn counting methods such as:

$$ SOC(t) = SOC(t-1) + \frac{I(t)}{Q_n} \Delta t $$

for a constant load, this reduces to:

$$ SOC = SOC_{initial} - \frac{I_{load}}{Q_{rated}}  \Delta t $$




For a step based iteration, the time difference $\Delta t$ is :

$$ \Delta t = n_{steps} \cdot \Delta t_{step} $$


If we define the charge consummed to be:

$$ Q_{consumed} = I_{load} \cdot \Delta t = I_{load} \cdot n_{steps} \cdot \Delta t_{step} $$


And starting from a fully charged battery, $SOC_{initial} = 1$, then the constant load state of charge equation becomes:

$$ SoC = 1 - \frac{Q_{consumed}}{Q_{rated}}$$

or:

$$ SoC = \frac{Q_{rated} - Q_{consumed}}{Q_{rated}}$$






### Analysis 1: 0.46A constant load for 150 minutes at 1 minute intervals




In [ ]:
# Case Values
I_load = 0.46

steps = 150.0;
time_step_sec = 60.0;
time_step_hr = time_step_sec/3600.0;


#### Compute State of Charge

As noted before

$$ Q_{consumed} = I_{load} \cdot n_{steps} \cdot \Delta t_{step} $$

$$ SoC = \frac{Q_{rated} - Q_{consumed}}{Q_{rated}}$$





In [ ]:
# compute consummed charge, amp hours
Q_consummed = steps * I_load * time_step_hr;

# compute state of chage in percentage
soc = (Q_rated - Q_consummed)/Q_rated;

#### Compute Internal Resistance

From the equation above where:
$$ R_{internal} = R_{internal}(SoC) $$

Where:

$ R_{internal},\ Internal\ Resistance\ of\ the\ Battery $

In [ ]:
# compute internal resistance
R_internal = polyval(p_ir,soc)

#### Voltage/IR Drop Between Load and Ideal Battery Source

Voltage source is the combination of the battery's ideal voltage source, the battery's internal resistance and the wiring harness to the load

The voltage drop between load and ideal source is the voltage drop across the battery's internal resistance and wiring harness, reffered to as the IR drop:

$$ (I \cdot R)_{drop} =  (R_{internal} + R_{harness}) \cdot I_{load} $$

In [ ]:
#compute IR drop
IR_drop = (R_internal + R_harness) * I_load

#### Compute Open-Circuit Voltage
$$ OCV_{cell}(Q_{consummed}) = a_0 + a_1 \cdot Q_{consummed} + a_2 \cdot Q_{consummed}^2 + ... + a_n \cdot Q_{consummed}^n $$


In [ ]:
# compute open circuit voltage
OCV_cell = polyval(p_dc,Q_consummed);

#### Voltage seen by Load (Voltage with Sag)

From Kirchoff's Voltage Law for a load attached to a battery back with a harness the voltage seen at the load is:

$$ V_{load} = OCV_{pack} - (I \cdot R)_{drop}$$

where

$$ OCV_{pack} = OCV_{cell} \cdot n_{cells} $$



In [ ]:
OCV_pack = OCV_cell * num_cells

V_load = OCV_pack - IR_drop

# Results:

In [ ]:
# output
print('0.46A constant load for 150 minutes at 1 minute intervals')
print('Charge Consummed[C] = ' + str(Q_consummed))
print('Open Circuit Cell Voltage[V] = ' + str(OCV_cell))
print('Open Circuit Pack Voltage[V] = ' + str(OCV_pack))
print('State of Charge[%] = ' + str(soc * 100))
print('Internal Resistance[Ohm] = ' + str( R_internal ))
print('Pack Voltage with sag[V] = ' + str( V_load ))

# Appendix: Extra Work

Following the procedure as noted above, similar can be found:


### Analysis 2: 0.46A constant load for 300 minutes at 1 minute intervals

In [ ]:
# Case 2:
I_load = 0.46

steps = 300.0;
time_step_sec = 60.0;
time_step_hr = time_step_sec/3600.0;

In [ ]:
# compute consummed charge, amp hours
Q_consummed = steps * I_load * time_step_hr;

# compute state of chage in percentage
soc = (Q_rated - Q_consummed)/Q_rated;

# compute internal resistance
R_internal = polyval(p_ir,soc);

#compute IR drop
IR_drop = (R_internal + R_harness) * I_load;

# compute open circuit voltage
OCV_cell = polyval(p_dc,Q_consummed);
OCV_pack = OCV_cell * num_cells

# compute load seen by voltage
V_load = OCV_pack - IR_drop

# output
print('0.46A constant load for 300 minutes at 1 minute intervals')
print('Charge Consummed[C] = ' + str(Q_consummed))
print('Open Circuit Cell Voltage[V] = ' + str(OCV_cell))
print('Open Circuit Pack Voltage[V] = ' + str(OCV_pack))
print('State of Charge[%] = ' + str(soc * 100))
print('Internal Resistance[Ohm] = ' + str( R_internal ))
print('Pack Voltage with sag[V] = ' + str( V_load ))

### Analysis 3: 15.0A constant load for 4 minutes at 1 minute intervals

In [ ]:
# Case 3:
I_load = 15.0

steps = 4.0;
time_step_sec = 60.0;
time_step_hr = time_step_sec/3600.0;

In [ ]:
# compute consummed charge, amp hours
Q_consummed = steps * I_load * time_step_hr;

# compute state of chage in percentage
soc = (Q_rated - Q_consummed)/Q_rated;

# compute internal resistance
R_internal = polyval(p_ir,soc);

#compute IR drop
IR_drop = (R_internal + R_harness) * I_load;

# compute open circuit voltage
OCV_cell = polyval(p_dc,Q_consummed);
OCV_pack = OCV_cell * num_cells

# compute load seen by voltage
V_load = OCV_pack - IR_drop

# output
print('15.0A constant load for 4 minutes at 1 minute intervals')
print('Charge Consummed[C] = ' + str(Q_consummed))
print('Open Circuit Cell Voltage[V] = ' + str(OCV_cell))
print('Open Circuit Pack Voltage[V] = ' + str(OCV_pack))
print('State of Charge[%] = ' + str(soc * 100))
print('Internal Resistance[Ohm] = ' + str( R_internal ))
print('Pack Voltage with sag[V] = ' + str( V_load ))

### Analysis 4: 15.0A constant load for 7 minutes at 1 minute intervals

In [ ]:
# Case 4:
I_load = 15.0

steps = 7.0;
time_step_sec = 60.0;
time_step_hr = time_step_sec/3600.0;

In [ ]:
# compute consummed charge, amp hours
Q_consummed = steps * I_load * time_step_hr;

# compute state of chage in percentage
soc = (Q_rated - Q_consummed)/Q_rated;

# compute internal resistance
R_internal = polyval(p_ir,soc);

#compute IR drop
IR_drop = (R_internal + R_harness) * I_load;

# compute open circuit voltage
OCV_cell = polyval(p_dc,Q_consummed);
OCV_pack = OCV_cell * num_cells

# compute load seen by voltage
V_load = OCV_pack - IR_drop

# output
print('15.0A constant load for 7 minutes at 1 minute intervals')
print('Charge Consummed[C] = ' + str(Q_consummed))
print('Open Circuit Cell Voltage[V] = ' + str(OCV_cell))
print('Open Circuit Pack Voltage[V] = ' + str(OCV_pack))
print('State of Charge[%] = ' + str(soc * 100))
print('Internal Resistance[Ohm] = ' + str( R_internal ))
print('Pack Voltage with sag[V] = ' + str( V_load ))

### Analysis 5: 35.0A constant load for 100 seconds at 1 second intervals

In [ ]:
# Case 5:
I_load = 35.0

steps = 100.0;
time_step_sec = 1.0;
time_step_hr = time_step_sec/3600.0;

In [ ]:
# compute consummed charge, amp hours
Q_consummed = steps * I_load * time_step_hr;

# compute state of chage in percentage
soc = (Q_rated - Q_consummed)/Q_rated;

# compute internal resistance
R_internal = polyval(p_ir,soc);

#compute IR drop
IR_drop = (R_internal + R_harness) * I_load;

# compute open circuit voltage
OCV_cell = polyval(p_dc,Q_consummed);
OCV_pack = OCV_cell * num_cells

# compute load seen by voltage
V_load = OCV_pack - IR_drop

# output
print('35.0A constant load for 100 seconds at 1 second intervals')
print('Charge Consummed[C] = ' + str(Q_consummed))
print('Open Circuit Cell Voltage[V] = ' + str(OCV_cell))
print('Open Circuit Pack Voltage[V] = ' + str(OCV_pack))
print('State of Charge[%] = ' + str(soc * 100))
print('Internal Resistance[Ohm] = ' + str( R_internal ))
print('Pack Voltage with sag[V] = ' + str( V_load ))


### Analysis 6: 35.0A constant load for 180 seconds at 1 second intervals

In [ ]:
# Case 6:
I_load = 35.0

steps = 180.0;
time_step_sec = 1.0;
time_step_hr = time_step_sec/3600.0;

In [ ]:
# compute consummed charge, amp hours
Q_consummed = steps * I_load * time_step_hr;

# compute state of chage in percentage
soc = (Q_rated - Q_consummed)/Q_rated;

# compute internal resistance
R_internal = polyval(p_ir, soc);

#compute IR drop
IR_drop = (R_internal + R_harness) * I_load;

# compute open circuit voltage
OCV_cell = polyval(p_dc,Q_consummed);
OCV_pack = OCV_cell * num_cells

# compute load seen by voltage
V_load = OCV_pack - IR_drop

# output
print('35.0A constant load for 180 seconds at 1 second intervals')
print('Charge Consummed[C] = ' + str(Q_consummed))
print('Open Circuit Cell Voltage[V] = ' + str(OCV_cell))
print('Open Circuit Pack Voltage[V] = ' + str(OCV_pack))
print('State of Charge[%] = ' + str(soc * 100))
print('Internal Resistance[Ohm] = ' + str( R_internal ))
print('Pack Voltage with sag[V] = ' + str( V_load ))